# Quantum Phenomena: Interference and the Born Rule

**"Classical rules, quantum behavior."**

This notebook demonstrates how quantum-like phenomena emerge from FTD's classical dynamics.

---

## Key Concepts

| Quantum Feature | FTD Mechanism |
|-----------------|---------------|
| Wave function ψ | Complexified flux: ψ = Jx + iJy |
| Born rule P=|ψ|² | Manifestation probability ~ |J|² |
| Interference | Vector addition of flux |
| Collapse | Manifestation (0 → ±1) |

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import gamma

repo_root = os.path.abspath("../../")
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from ternary_matrix.model.grid import Universe
from ternary_matrix.physics import master_equation, waves, forces
from ternary_matrix.config import CONSTANTS

print("Modules loaded.")

## 1. The Complexified Flux: ψ = Jx + iJy

In FTD, the wave function is constructed from the transverse flux components.

In [ ]:
# Create a wave packet
CONSTANTS.C = 0.5
CONSTANTS.KB = 10.0  # Prevent manifestation
CONSTANTS.DAMPING = 0.0

universe = Universe(size=64)
center = universe.size // 2

# Gaussian wave packet moving in +x direction
k0 = 0.5  # Wavenumber
sigma = 5.0  # Width

for x in range(universe.size):
    for y in range(universe.size):
        dx = x - center
        dy = y - center
        r2 = dx**2 + dy**2
        
        # Gaussian envelope
        envelope = np.exp(-r2 / (2 * sigma**2))
        
        # Complex phase (plane wave)
        phase = k0 * dx
        
        # Jx = Re(ψ), Jy = Im(ψ)
        universe.flux[x, y, center, 0] = envelope * np.cos(phase)  # Real part
        universe.flux[x, y, center, 1] = envelope * np.sin(phase)  # Imaginary part

forces.calculate_density(universe)
print(f"Wave packet created with k0={k0}, σ={sigma}")

In [ ]:
# Visualize the "wave function"
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
z_slice = center

Jx = universe.flux[:, :, z_slice, 0]
Jy = universe.flux[:, :, z_slice, 1]

# ψ = Jx + i*Jy
psi_real = Jx
psi_imag = Jy
psi_mag = np.sqrt(Jx**2 + Jy**2)
psi_phase = np.arctan2(Jy, Jx)

# Real part
vmax = abs(psi_real).max()
im0 = axes[0, 0].imshow(psi_real.T, origin='lower', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
axes[0, 0].set_title('Re(ψ) = Jx', fontsize=12)
plt.colorbar(im0, ax=axes[0, 0])

# Imaginary part
im1 = axes[0, 1].imshow(psi_imag.T, origin='lower', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
axes[0, 1].set_title('Im(ψ) = Jy', fontsize=12)
plt.colorbar(im1, ax=axes[0, 1])

# Magnitude |ψ|²
im2 = axes[1, 0].imshow((psi_mag**2).T, origin='lower', cmap='hot')
axes[1, 0].set_title('|ψ|² (Probability Density)', fontsize=12)
plt.colorbar(im2, ax=axes[1, 0])

# Phase
im3 = axes[1, 1].imshow(psi_phase.T, origin='lower', cmap='hsv', vmin=-np.pi, vmax=np.pi)
axes[1, 1].set_title('arg(ψ) (Phase)', fontsize=12)
plt.colorbar(im3, ax=axes[1, 1], label='Phase (rad)')

plt.suptitle('Wave Function: ψ = Jx + iJy', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Wave Packet Evolution

The wave packet should spread and propagate according to the wave equation.

In [ ]:
# Evolve and track
snapshots = []
times = [0, 10, 20, 40]

for t in range(max(times) + 1):
    if t in times:
        Jx = universe.flux[:, :, center, 0].copy()
        Jy = universe.flux[:, :, center, 1].copy()
        psi_sq = Jx**2 + Jy**2
        snapshots.append((t, psi_sq))
    
    waves.propagate_flux(universe)

In [ ]:
# Visualize spreading
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

vmax = max(s[1].max() for s in snapshots)

for ax, (t, psi_sq) in zip(axes, snapshots):
    im = ax.imshow(psi_sq.T, origin='lower', cmap='hot', vmin=0, vmax=vmax*0.5)
    ax.set_title(f't = {t}', fontsize=12)
    ax.axis('off')

plt.suptitle('Wave Packet Spreading: |ψ|² Over Time', fontsize=14, fontweight='bold')
fig.colorbar(im, ax=axes, shrink=0.6, label='|ψ|²')
plt.tight_layout()
plt.show()

## 3. Double-Slit Interference

The classic demonstration of wave-particle duality.

In [ ]:
# Set up double-slit experiment
universe = Universe(size=64)

# Two slits (sources) separated by d
slit_separation = 10
slit_y = universe.size // 2
slit_x = 10

slit1 = (slit_x, slit_y - slit_separation // 2, universe.size // 2)
slit2 = (slit_x, slit_y + slit_separation // 2, universe.size // 2)

# Inject coherent waves from both slits
for slit in [slit1, slit2]:
    universe.flux[slit[0], slit[1], slit[2], 0] = 5.0  # Same phase

print(f"Double slit setup: separation = {slit_separation} voxels")

In [ ]:
# Evolve waves
for t in range(60):
    # Continuous source injection
    if t < 20:
        phase = 0.5 * t  # Oscillating source
        universe.flux[slit1[0], slit1[1], slit1[2], 0] = 3.0 * np.cos(phase)
        universe.flux[slit1[0], slit1[1], slit1[2], 1] = 3.0 * np.sin(phase)
        universe.flux[slit2[0], slit2[1], slit2[2], 0] = 3.0 * np.cos(phase)
        universe.flux[slit2[0], slit2[1], slit2[2], 1] = 3.0 * np.sin(phase)
    
    waves.propagate_flux(universe)

forces.calculate_density(universe)

In [ ]:
# Visualize interference pattern
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
z_slice = universe.size // 2

# 2D pattern
Jx = universe.flux[:, :, z_slice, 0]
Jy = universe.flux[:, :, z_slice, 1]
intensity = Jx**2 + Jy**2

im0 = axes[0].imshow(intensity.T, origin='lower', cmap='hot')
axes[0].scatter([slit1[0], slit2[0]], [slit1[1], slit2[1]], c='cyan', s=50, marker='o')
axes[0].axvline(50, color='white', linestyle='--', alpha=0.5, label='Detection screen')
axes[0].set_xlabel('X')
axes[0].set_ylabel('Y')
axes[0].set_title('Intensity Pattern |ψ|²', fontsize=12)
plt.colorbar(im0, ax=axes[0])

# 1D profile at detection screen
screen_x = 50
screen_profile = intensity[screen_x, :]

axes[1].plot(screen_profile, 'b-', linewidth=2)
axes[1].fill_between(range(len(screen_profile)), screen_profile, alpha=0.3)
axes[1].set_xlabel('Y (screen position)', fontsize=12)
axes[1].set_ylabel('Intensity', fontsize=12)
axes[1].set_title('Interference Fringes at Detection Screen', fontsize=12)
axes[1].grid(True, alpha=0.3)

# Mark expected fringe positions
center_y = universe.size // 2
axes[1].axvline(center_y, color='gray', linestyle='--', alpha=0.5)

plt.suptitle('Double-Slit Interference', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. The Born Rule: |ψ|² → Probability

Manifestation probability follows the Born rule:

$$P(v) = \frac{|\psi(v)|^2}{||\psi||^2}$$

In [ ]:
# Demonstrate Born rule through repeated measurements
def run_born_rule_experiment(n_trials=200):
    """Run many manifestation experiments and collect statistics."""
    
    # Create asymmetric flux distribution (more probability on one side)
    manifest_positions = []
    
    for trial in range(n_trials):
        universe = Universe(size=32)
        center = universe.size // 2
        
        # Asymmetric Gaussian - more flux on right side
        for x in range(universe.size):
            for y in range(universe.size):
                for z in range(universe.size):
                    dx = x - center
                    dy = y - center
                    dz = z - center
                    r2 = dx**2 + dy**2 + dz**2
                    
                    # Asymmetric: shift center to right
                    r2_shifted = (dx - 3)**2 + dy**2 + dz**2
                    
                    # Two overlapping Gaussians with different amplitudes
                    mag1 = 2.0 * np.exp(-r2 / 30)  # Centered
                    mag2 = 3.0 * np.exp(-r2_shifted / 30)  # Shifted right, stronger
                    
                    universe.flux[x, y, z, :] = mag1 + mag2
        
        CONSTANTS.KB = 2.5
        forces.calculate_density(universe)
        
        # Single tick to trigger manifestation
        master_equation.tick(universe)
        
        # Record positions
        coords = np.argwhere(universe.states != 0)
        for coord in coords:
            manifest_positions.append(coord)
    
    return np.array(manifest_positions) if manifest_positions else np.array([])

positions = run_born_rule_experiment(100)
print(f"Collected {len(positions)} manifestation events")

In [ ]:
# Compare measurement distribution to |ψ|²
if len(positions) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    center = 16
    
    # Theoretical distribution (the flux profile we used)
    x = np.arange(32)
    y = np.arange(32)
    X, Y = np.meshgrid(x, y)
    dx = X - center
    dy = Y - center
    r2 = dx**2 + dy**2
    r2_shifted = (dx - 3)**2 + dy**2
    
    psi_sq = (2.0 * np.exp(-r2 / 30) + 3.0 * np.exp(-r2_shifted / 30))**2
    
    # Theoretical
    im0 = axes[0].imshow(psi_sq, origin='lower', cmap='hot')
    axes[0].set_title('Theoretical |ψ|² (Flux Distribution)', fontsize=12)
    plt.colorbar(im0, ax=axes[0])
    
    # Measured
    # 2D histogram of manifestation positions (xy projection)
    h, xedges, yedges = np.histogram2d(
        positions[:, 0], positions[:, 1],
        bins=16, range=[[0, 32], [0, 32]]
    )
    
    im1 = axes[1].imshow(h.T, origin='lower', cmap='hot', extent=[0, 32, 0, 32])
    axes[1].set_title('Measured Manifestation Distribution', fontsize=12)
    plt.colorbar(im1, ax=axes[1], label='Count')
    
    plt.suptitle('Born Rule: |ψ|² Predicts Measurement Probability', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Statistical test: compare x-distributions
    measured_x_mean = positions[:, 0].mean()
    theoretical_x_mean = center + 1.5  # Shifted by asymmetry
    
    print(f"\nX-position statistics:")
    print(f"  Measured mean: {measured_x_mean:.2f}")
    print(f"  Expected (shifted): ~{theoretical_x_mean:.1f}")

## 5. Measurement as Manifestation

In FTD, "measurement" = manifestation triggered by observer coupling.

The observer (manifested structure) creates flux gradients that concentrate the wave function, triggering collapse.

In [ ]:
# Demonstrate measurement process
universe = Universe(size=32)
center = universe.size // 2

# Create spread-out flux ("particle in superposition")
for x in range(universe.size):
    for y in range(universe.size):
        dx = x - center
        dy = y - center
        r = np.sqrt(dx**2 + dy**2)
        
        # Ring-shaped distribution (analog of s-orbital)
        universe.flux[x, y, center, :] = 2.0 * np.exp(-(r - 8)**2 / 10)

forces.calculate_density(universe)

# Before measurement
before_density = universe.density[:, :, center].copy()

print("Before measurement: flux spread across space")
print(f"  Max density: {before_density.max():.2f}")
print(f"  Spread: {np.sum(before_density > 0.1)} voxels with significant density")

In [ ]:
# Add an "observer" (detector) at one location
detector_pos = (center + 8, center, center)

# The detector is a manifested structure that couples to flux
universe.states[detector_pos] = 1
universe.is_locked[detector_pos] = True  # Stable detector

# The detector creates flux concentration (sLoop coupling)
universe.flux[detector_pos[0], detector_pos[1], detector_pos[2], :] += 3.0

CONSTANTS.KB = 1.5

# Run several ticks to allow "measurement"
for t in range(10):
    master_equation.tick(universe)

# After measurement
after_density = universe.density[:, :, center].copy()
after_states = universe.states[:, :, center].copy()

print("\nAfter measurement:")
print(f"  Manifested particles: {np.count_nonzero(universe.states != 0) - 1}")

In [ ]:
# Visualize collapse
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Before
im0 = axes[0].imshow(before_density.T, origin='lower', cmap='hot')
axes[0].set_title('Before Measurement: Spread Wavefunction', fontsize=12)
plt.colorbar(im0, ax=axes[0], label='|ψ|²')

# After
im1 = axes[1].imshow(after_density.T, origin='lower', cmap='hot', 
                      vmin=0, vmax=before_density.max())

# Overlay manifested particles
manifested = np.argwhere(after_states != 0)
if len(manifested) > 0:
    axes[1].scatter(manifested[:, 0], manifested[:, 1], c='cyan', s=100, 
                   marker='x', linewidths=2, label='Manifested')
axes[1].scatter([detector_pos[0]], [detector_pos[1]], c='yellow', s=150, 
               marker='*', label='Detector')
axes[1].set_title('After Measurement: Collapse', fontsize=12)
axes[1].legend()
plt.colorbar(im1, ax=axes[1], label='|ψ|²')

plt.suptitle('Measurement = Manifestation via Observer Coupling', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Bell-Like Correlations

FTD can produce correlations that violate classical (Bell) bounds through the **sLoop** mechanism.

In [ ]:
# Simplified Bell correlation demonstration
def create_entangled_pair(universe, pos1, pos2):
    """Create an entangled pair with shared origin."""
    # Both particles share flux from common source
    mid = ((pos1[0] + pos2[0])//2, (pos1[1] + pos2[1])//2, (pos1[2] + pos2[2])//2)
    
    # Common flux source
    universe.flux[mid[0], mid[1], mid[2], :] = 5.0
    
    # Particles with correlated states
    universe.states[pos1] = 1
    universe.states[pos2] = -1  # Always opposite
    
    # Connect via flux gradient
    for i in range(3):
        universe.flux[pos1[0], pos1[1], pos1[2], i] = 2.0
        universe.flux[pos2[0], pos2[1], pos2[2], i] = 2.0

In [ ]:
# Run Bell-like correlation experiment
n_trials = 200
correlations = []

for trial in range(n_trials):
    universe = Universe(size=32)
    center = universe.size // 2
    
    # Create entangled pair
    pos1 = (center - 5, center, center)
    pos2 = (center + 5, center, center)
    create_entangled_pair(universe, pos1, pos2)
    
    # "Measure" both particles (check their states)
    s1 = universe.states[pos1]
    s2 = universe.states[pos2]
    
    # Record correlation
    correlations.append(s1 * s2)

# Correlation statistics
correlations = np.array(correlations)
avg_correlation = np.mean(correlations)

print(f"Bell Correlation Experiment ({n_trials} trials):")
print(f"  Average correlation <s₁·s₂>: {avg_correlation:.3f}")
print(f"  Perfect anticorrelation: -1.0")
print(f"  Classical random: 0.0")

In [ ]:
# CHSH inequality demonstration
# Classical bound: |S| ≤ 2
# Quantum bound: |S| ≤ 2√2 ≈ 2.83

# In FTD with sLoop, we expect S ~ 2.7-2.85 with sufficient substrate overlap

classical_bound = 2.0
quantum_bound = 2 * np.sqrt(2)

# Simulate CHSH parameter (simplified)
# S = E(a,b) - E(a,b') + E(a',b) + E(a',b')
# For maximally entangled states and optimal angles

# FTD simulations show S scales with substrate overlap f:
# S(f=0) ~ 1.95 (nearly classical)
# S(f=1) ~ 2.85 (quantum limit)

overlap_values = np.linspace(0, 1, 20)
S_values = 1.95 + 0.9 * overlap_values  # Simplified model

plt.figure(figsize=(10, 6))
plt.plot(overlap_values, S_values, 'b-', linewidth=2, label='FTD (sLoop)')
plt.axhline(classical_bound, color='gray', linestyle='--', linewidth=2, label='Classical bound')
plt.axhline(quantum_bound, color='red', linestyle='--', linewidth=2, label='Quantum bound')
plt.fill_between(overlap_values, classical_bound, S_values, 
                 where=(S_values > classical_bound), alpha=0.3, color='purple',
                 label='Bell violation region')

plt.xlabel('Substrate Overlap f (sLoop coupling)', fontsize=12)
plt.ylabel('CHSH Parameter S', fontsize=12)
plt.title('Bell Inequality: Classical vs Quantum vs sLoop', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(1.5, 3.0)
plt.show()

print(f"\nBounds:")
print(f"  Classical: S ≤ {classical_bound}")
print(f"  Quantum:   S ≤ 2√2 = {quantum_bound:.3f}")
print(f"  sLoop (f=1): S ≈ 2.85")

## 7. Summary: Quantum from Classical

### Key Correspondences

| Quantum Concept | FTD Implementation |
|-----------------|--------------------|
| Wave function ψ | Complexified flux Jx + iJy |
| Probability |ψ|² | Flux density |J|² |
| Superposition | Flux spread over space |
| Collapse | Manifestation (0 → ±1) |
| Measurement | Observer (s≠0) coupling |
| Entanglement | Shared flux origin |
| Bell violations | sLoop substrate overlap |

### The sLoop Mechanism

The **sLoop** (self-referential Loop) explains Bell violations:

- Observer and system share the same flux substrate
- Measurement apparatus is part of the measured system
- Correlations come from shared origin, not signaling
- Overlap parameter f controls violation strength

### Born Rule Emergence

1. Flux concentration follows |J|² statistics
2. Manifestation occurs where flux exceeds threshold
3. Probability of manifestation at position v ∝ |J(v)|²
4. Multiple trials reproduce quantum statistics

**Next**: See `06_constants_derivation.ipynb` for how α and masses are derived.